In [0]:
# Here all the Imports and Path Constants will be defined
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField, 
    StringType, FloatType,
    TimestampType, IntegerType
)

# Paths - 
VOLUME_BASE =  "/Volumes/workspace/default/iot_pipeline"
RAW_DIR = f"{VOLUME_BASE}/raw" # Here all of the 4 CSV files will be live
CLEARN_DIR = f"{VOLUME_BASE}/clean" # Here all of the output files will go

# Threshold for cleaning - 
TEMP_MIN = -40.0 # The physical plausible temperature is between -40 and 85
TEMP_MAX = 85.0 # Our injected faults are between -999 and 9999 are both outside ranges. 

print("Imports and constants loaded")
print(f"Reading from {RAW_DIR}")
print(f"Writing to {CLEARN_DIR}")
print(f"Valid temperatures are from {TEMP_MIN}°C to {TEMP_MAX}°C")





Imports and constants loaded
Reading from /Volumes/workspace/default/iot_pipeline/raw
Writing to /Volumes/workspace/default/iot_pipeline/clean
Valid temperatures are from -40.0°C to 85.0°C


In [0]:
# Here we are defining schema as every column gets a name, and nullability flag, and here spark reads this before opening a single CSV file.

schema = StructType([
    # The syntax here is defined in a way as StructField(column_name, data_type, nullable), if nullable = True which means this columns contains null other it should have a value. 

    StructField("device_id", StringType(), True),
    StructField("timestamp", StringType(), True),
    StructField("temperature_c", FloatType(), True),
    StructField("humidity_pct", FloatType(), True),
    StructField("region", StringType(), False),
    StructField("Status", StringType(), False),
    StructField("battery_level", FloatType(), True),
    StructField("event_type", StringType(), False),
])
    
# Print the schema to the console as the spark will sees it
print(f" Total Columns: {len(schema.fields)}")
print()
for field in schema.fields:
    nullable_label = "nullable" if field.nullable else "NOT NULL"
    print(f"{field.name:<20} {str(field.dataType):<15} {nullable_label}")

print()
print(" 'INVALID_DATE' strings survive ingestiong here")


 Total Columns: 8

device_id            StringType()    nullable
timestamp            StringType()    nullable
temperature_c        FloatType()     nullable
humidity_pct         FloatType()     nullable
region               StringType()    NOT NULL
Status               StringType()    NOT NULL
battery_level        FloatType()     nullable
event_type           StringType()    NOT NULL

 'INVALID_DATE' strings survive ingestiong here


In [0]:
# Here we are reading the RAW CSV's using the explicit schema.
# Here for the first time, Spark owns the 4 CSV files for the first time, here spark does reads enough to validate only, and does not scan all the 12 million rows. The full row scanning will be done when we take action calls such as .show() or the .count()

import time

t_start = time.time()

df_ingested = (
    spark.read
    .option("header", True) # Here while reading the data, we are telling spark that the first row is the header, and not the actual data
    .option("mode", "PERMISSIVE") # Here we are telling spark that if it finds any error while reading the data, it should not stop, but should continue reading the data, and report the error later
    .schema(schema) # Here we are passing the schema that we defined above
    .csv(RAW_DIR) # Here we reading all the 4 files, in the folder as one logical Dataframe
)

t_end = time.time()

print(f" Read operations defined in {t_end - t_start:.3f} seconds")
print(f" Still we are lazy mode, and spark has not read the data yet, it is just defined the read operation")
df_ingested.printSchema()


 Read operations defined in 0.001 seconds
 Still we are lazy mode, and spark has not read the data yet, it is just defined the read operation
root
 |-- device_id: string (nullable = true)
 |-- timestamp: string (nullable = true)
 |-- temperature_c: float (nullable = true)
 |-- humidity_pct: float (nullable = true)
 |-- region: string (nullable = true)
 |-- Status: string (nullable = true)
 |-- battery_level: float (nullable = true)
 |-- event_type: string (nullable = true)



In [0]:
# Here we are cleaning the data, as we have to cast Timestamp, and filtering the malformed Rows, here we have to ingest the dataframe which is generally (RAW, DIRTY), it becomes clean dataframe (validated). Every transformation here directly targets an error which we have to fix

df_clean = (
    df_ingested
    
    .withColumn(
        "timestamp_parsed", 
        F.try_to_timestamp(F.col("timestamp"), F.lit("yyyy-MM-dd HH:mm:ss"))
        # This is where we are explicitely defining the format of the timestamp column, if the string matches then it will be valid TimeStampType, if not then it will become null, and here the null is intetional as well as controlled. 

    )

    .filter(F.col("device_id").isNotNull())
    # Here we will remove the 0.5% null device_id rows which is injected and here we dropping because if the sensor reading no device identity, and then it cannot be attributed to any device. 

    .filter(F.col("timestamp_parsed").isNotNull())
    .filter(
        (F.col("temperature_c")>=TEMP_MIN) &
        (F.col("temperature_c") <= TEMP_MAX)
    )

    .drop("timestamp") #Here we are dropping the original string column as we don't needed both versions, and keeping the both will confuse us while reading the dataframe later. 

    .withColumnRenamed("timestamp_parsed", "timestamp")
    # Here we are getting the cleaned columned back to timestamp
)
print("Cleaning logic defined")
df_clean.printSchema()








    
    
    

Cleaning logic defined
root
 |-- device_id: string (nullable = true)
 |-- temperature_c: float (nullable = true)
 |-- humidity_pct: float (nullable = true)
 |-- region: string (nullable = true)
 |-- Status: string (nullable = true)
 |-- battery_level: float (nullable = true)
 |-- event_type: string (nullable = true)
 |-- timestamp: timestamp (nullable = true)



In [0]:
# Here for the first time we will scans all of the 12 million rows and executes the filtering logic for the execution
import time
t_start = time.time()

raw_count = df_ingested.count()
# here we are counting every row before filters, this includes all of the garbage rows as well. 

t_mid = time.time()

clean_count = df_clean.count()
# here we are counting every row after filters, such as device_id, timestamp, and temperature validation all applied. 
t_end = time.time()

dropped = raw_count - clean_count
dropped_pct = (dropped / raw_count) * 100

print(f"Raw row count :{raw_count}")
print(f"Clean row count :{clean_count}")
print(f"Dropped row count :{dropped:,} ({dropped_pct:.2f}%)")
print(f"Raw count time : {t_mid - t_start:.1f} seconds")
print(f"Clean count time : {t_end - t_mid:.1f} seconds")



Raw row count :12000000
Clean row count :11468644
Dropped row count :531,356 (4.43%)
Raw count time : 13.2 seconds
Clean count time : 6.2 seconds


In [0]:
# Now this is mostly used for to display the cleaned sample
df_clean.show(10, truncate = False)
# Here we are showing rows = 10, and truncate = False means don't cutoff long values, and here we want to see full timestamp, and region string clearly, and we are also confirming zero nulls remain in our critical columns. 
null_check = df_clean.select(
    F.sum(F.col("device_id").isNull().cast("int")).alias("null_device_id"),
    F.sum(F.col("timestamp").isNull().cast("int")).alias("null_timestamp"),
)
null_check.show()


+-----------+-------------+------------+------+--------+-------------+------------------+-------------------+
|device_id  |temperature_c|humidity_pct|region|Status  |battery_level|event_type        |timestamp          |
+-----------+-------------+------------+------+--------+-------------+------------------+-------------------+
|SENSOR_0004|53.229404    |46.717644   |West  |OFFLINE |6.0297995    |TEMP_READING      |2024-03-09 00:08:15|
|SENSOR_0008|5.417027     |46.746193   |North |WARN    |5.1939993    |TEMP_READING      |2024-01-13 18:13:28|
|SENSOR_0012|34.89139     |72.84301    |North |CRITICAL|56.822548    |BATTERY_LOW       |2024-08-12 08:41:43|
|SENSOR_0016|36.506752    |12.696333   |East  |OFFLINE |55.427788    |THRESHOLD_BREACHED|2024-08-20 23:29:00|
|SENSOR_0020|40.868423    |26.756752   |East  |CRITICAL|15.7652855   |BATTERY_LOW       |2024-08-22 01:54:19|
|SENSOR_0024|-9.742063    |91.35516    |East  |WARN    |23.032478    |HEARTBEAT         |2024-01-31 01:07:19|
|SENSOR_00

In [0]:
# Here we are Extracting time components from Timestamp
# Before we can aggregate "events per hour" or "Average time per day"
# We needed hour/day/month as their own columns

df_timed = (
    df_clean
    .withColumn("event_hour", F.hour(F.col("timestamp")))
    # F.hour() extracts just the component from the Timestamp column here
    # This is per-row operation which runs independantly and here we do not need to shuffle the rows, extracting a value from one row never needs data from other row

    .withColumn("event_day", F.dayofmonth(F.col("timestamp")))
    .withColumn("event_month", F.month(F.col("timestamp")))
    
)

df_timed.select("timestamp","event_hour","event_day","event_month").show(10, truncate = False)


+-------------------+----------+---------+-----------+
|timestamp          |event_hour|event_day|event_month|
+-------------------+----------+---------+-----------+
|2024-03-09 00:08:15|0         |9        |3          |
|2024-01-13 18:13:28|18        |13       |1          |
|2024-08-12 08:41:43|8         |12       |8          |
|2024-08-20 23:29:00|23        |20       |8          |
|2024-08-22 01:54:19|1         |22       |8          |
|2024-01-31 01:07:19|1         |31       |1          |
|2024-12-30 14:09:38|14        |30       |12         |
|2024-04-26 21:56:28|21        |26       |4          |
|2024-03-16 20:13:26|20        |16       |3          |
|2024-12-21 23:43:01|23        |21       |12         |
+-------------------+----------+---------+-----------+
only showing top 10 rows


In [0]:
# import time
df_region_summary =  (
    df_timed
    .groupBy("region")
    .agg(
        F.count("*").alias("total_events"),
        F.avg("temperature_c").alias("avg_temperature_c"),
        F.avg("humidity_pct").alias("avg_humidity_pct"),
        F.avg("battery_level").alias("avg_battery_level"),
        F.countDistinct("device_id").alias("unique_devices"),
    )
    .orderBy(F.col("total_events").desc())
)

print()

t_start = time.time()
display(df_region_summary)
t_end = time.time()
print(f"Shuffle + Aggregation + Sort time : {t_end - t_start:.1f} seconds")

region,total_events,avg_temperature_c,avg_humidity_pct,avg_battery_level,unique_devices
North,2869187,22.48994997627723,50.01386950334658,49.98661113544666,1000
West,2868067,22.50387262932706,49.968702654315884,50.01383474252853,1000
East,2866681,22.502881924142446,49.99529921422209,50.004891069696754,1000
South,2864709,22.498581102177912,50.00566580853664,49.998644433671316,1000


Shuffle + Aggregation + Sort time : 5.6 seconds


In [0]:
df_hourly_summary = (
    df_timed
    .groupBy("region", "event_hour")
    .agg(
        F.avg("temperature_c").alias("avg_temp_c"),
        F.count("*").alias("reading_count"),
    )
    .orderBy("region", "event_hour")
)

df_hourly_summary.show(12, truncate = False)


+------+----------+------------------+-------------+
|region|event_hour|avg_temp_c        |reading_count|
+------+----------+------------------+-------------+
|East  |0         |22.63030181284181 |119363       |
|East  |1         |22.502187286889544|119625       |
|East  |2         |22.520014212502986|119393       |
|East  |3         |22.422641116621207|119883       |
|East  |4         |22.534436365323646|119167       |
|East  |5         |22.523601305675903|118860       |
|East  |6         |22.517587481559026|119189       |
|East  |7         |22.56572297057349 |119885       |
|East  |8         |22.504883710012678|119103       |
|East  |9         |22.5264137723971  |119464       |
|East  |10        |22.439460905077738|119478       |
|East  |11        |22.531068644071453|119191       |
+------+----------+------------------+-------------+
only showing top 12 rows


In [0]:
# Here we are ranking devices by average temperature within each region using a window function
# Three-step process:
#   1. Group by device_id + region to get one row per device with its average temperature
#   2. Define a window: partition by region, order by avg temp descending
#   3. Apply F.rank() over that window — every row keeps existing, we just add a rank column
#   4. Filter to top 3 hottest devices per region (temp_rank <= 3)

from pyspark.sql import Window

# Step 1: One row per device per region with its average temperature
df_device_avg = (
    df_clean
    .groupBy("device_id", "region")
    .agg(
        F.avg("temperature_c").alias("avg_temp_c"),
        F.count("*").alias("reading_count"),
    )
)

# Step 2: Define the window — partition by region, order by avg temp descending
region_temp_window = Window.partitionBy("region").orderBy(F.col("avg_temp_c").desc())

# Step 3: Apply rank over the window, keeping every row and adding a rank column
# Step 4: Filter to top 3 hottest devices per region
df_ranked = (
    df_device_avg
    .withColumn("temp_rank", F.rank().over(region_temp_window))
    .filter(F.col("temp_rank") <= 3)  # keep only the top 3 hottest devices per region
    .orderBy("region", "temp_rank")
)

df_ranked.show(20, truncate = False)

+-----------+------+------------------+-------------+---------+
|device_id  |region|avg_temp_c        |reading_count|temp_rank|
+-----------+------+------------------+-------------+---------+
|SENSOR_0999|East  |23.682571543763075|2848         |1        |
|SENSOR_0812|East  |23.55221228070682 |2835         |2        |
|SENSOR_0698|East  |23.473466846321397|2890         |3        |
|SENSOR_0107|North |23.490109972673515|2887         |1        |
|SENSOR_0842|North |23.44049138673101 |2885         |2        |
|SENSOR_0206|North |23.435378482713634|2938         |3        |
|SENSOR_0567|South |23.715685736725924|2877         |1        |
|SENSOR_0747|South |23.694572110315384|2825         |2        |
|SENSOR_0266|South |23.632680649840506|2880         |3        |
|SENSOR_0678|West  |23.9791552149091  |2842         |1        |
|SENSOR_0079|West  |23.683143133314537|2743         |2        |
|SENSOR_0712|West  |23.61090334557339 |2769         |3        |
+-----------+------+------------------+-

In [0]:
# ─── Cell 11: Write Region Summary as Partitioned Parquet ────────────────────
# This is the Milestone 4 deliverable — optimized columnar storage,
# partitioned by region, with exactly 1 file per partition folder.

import time

OUTPUT_DIR = f"{VOLUME_BASE}/output"

t_start = time.time()

(
    df_region_summary
    .coalesce(1)
    # ↑ Merges all internal Spark partitions into ONE partition BEFORE writing.
    # Why here and not earlier? We want Spark to do the shuffle and aggregation
    # across all available parallelism first — coalesce only at the last step,
    # the write, where we care about output file count not computation speed.
    # coalesce(1) does NOT mean "put everything in one file" — it means
    # "each partitionBy group gets 1 file." Combined with partitionBy("region")
    # below, the result is 1 file per region folder, not 1 file total.

    .write
    .mode("overwrite")
    .partitionBy("region")
    # ↑ Creates one subfolder per unique region value:
    #   output/region_summary/region=East/
    #   output/region_summary/region=North/  etc.
    # Spark REMOVES the "region" column from the Parquet file itself —
    # the region value is encoded in the folder name instead.
    # Downstream readers reconstruct the column automatically from the path.

    .parquet(f"{OUTPUT_DIR}/region_summary")
    # .parquet() instead of .csv() — columnar, compressed, schema-embedded.
    # No header option needed — schema lives in the file footer, not a header row.
)

t_end = time.time()
print(f"✅ Region summary written in {t_end - t_start:.1f} seconds")

# ── Verify the folder structure ───────────────────────────────────────────────
print("\n📁 Output folder structure:")
for f in dbutils.fs.ls(f"{OUTPUT_DIR}/region_summary"):
    print(f"  📂 {f.name}")
    for inner in dbutils.fs.ls(f.path):
        size_kb = inner.size / 1024
        print(f"       📄 {inner.name:<60} {size_kb:.1f} KB")

✅ Region summary written in 8.4 seconds

📁 Output folder structure:
  📂 _SUCCESS
       📄 _SUCCESS                                                     0.0 KB
  📂 _committed_7115861979401610363
       📄 _committed_7115861979401610363                               0.0 KB
  📂 region=East/
       📄 _SUCCESS                                                     0.0 KB
       📄 _committed_1662893452379641461                               0.1 KB
       📄 _committed_7115861979401610363                               0.2 KB
       📄 _started_1662893452379641461                                 0.0 KB
       📄 _started_7115861979401610363                                 0.0 KB
       📄 part-00000-tid-7115861979401610363-0e86e226-ab87-4c0e-b8a3-bf57d0295b19-251-1.c000.snappy.parquet 1.7 KB
  📂 region=North/
       📄 _SUCCESS                                                     0.0 KB
       📄 _committed_1662893452379641461                               0.1 KB
       📄 _committed_7115861979401610363   

In [0]:
# ─── Cell 12: Write Hourly Summary as Partitioned Parquet ────────────────────
# Same pattern as Cell 11, applied to the 96-row hourly aggregation.
# Partitioned by region; each region folder holds its 24 hourly rows.

import time

t_start = time.time()

(
    df_hourly_summary
    .coalesce(1)
    # Merge internal partitions to one before writing, so each region
    # folder gets exactly one file.

    .write
    .mode("overwrite")
    .partitionBy("region")
    # One folder per region, with the region column encoded in the path.

    .parquet(f"{OUTPUT_DIR}/hourly_summary")
)

t_end = time.time()
print(f" Hourly summary written in {t_end - t_start:.1f} seconds")

# ── Verify structure ──────────────────────────────────────────────────────────
for f in dbutils.fs.ls(f"{OUTPUT_DIR}/hourly_summary"):
    if f.name.startswith("region="):
        inner = [x for x in dbutils.fs.ls(f.path) if x.name.endswith(".parquet")]
        print(f"   {f.name:<15} {len(inner)} parquet file(s)")

 Hourly summary written in 7.7 seconds
   region=East/    1 parquet file(s)
   region=North/   1 parquet file(s)
   region=South/   1 parquet file(s)
   region=West/    1 parquet file(s)


In [0]:
# ─── Cell 13: Read Back and Prove Partition Pruning ──────────────────────────
# Reading the Parquet back. The schema comes from the file footer, so no
# schema declaration is needed here, unlike the CSV in Milestone 2.

df_hourly_back = spark.read.parquet(f"{OUTPUT_DIR}/hourly_summary")

# 1. Full read: should return 96 rows (4 regions × 24 hours)
print(f"Total rows across all regions: {df_hourly_back.count()}")

# 2. Filtered read: only East. Spark should prune the other 3 folders.
df_east = df_hourly_back.filter(F.col("region") == "East")
print(f"East rows: {df_east.count()}")

# 3. Show the physical plan. Look for PartitionFilters in the output.
df_east.explain()

Total rows across all regions: 96
East rows: 24
== Physical Plan ==
PhotonResultStage
+- PhotonColumnarToRow
   +- PhotonScan parquet [event_hour#11511,avg_temp_c#11512,reading_count#11513L,region#11514] DataFilters: [], DictionaryFilters: [], Format: parquet, Location: InMemoryFileIndex(1 paths)[dbfs:/Volumes/workspace/default/iot_pipeline/output/hourly_summary], OptionalDataFilters: [], PartitionFilters: [isnotnull(region#11514), (region#11514 = East)], ReadSchema: struct<event_hour:int,avg_temp_c:double,reading_count:bigint>, RequiredDataFilters: []


== Photon Explanation ==
The query is fully supported by Photon.
